# SEVIRI SSL: training objectives and results

This notebook follows the [project specification](../handoff_codex_ssl_seviri.md) and compares four learning objectives: **MVM** reconstructs hidden historical observations; **Future** predicts future observations; **Joint** combines reconstruction and forecasting; **Residual** predicts a correction to an advective forecast. The completed historical grid includes VideoMAE-Small and VideoMAE-Base at **4,096, 8,192, 16,384, 32,768 and 65,536 clips**. The same objective figures include **Base Future on 131,072 clips**, completed on Leonardo, and the saved three-epoch Small MVM trajectory at that size.

**Apparent advection** describes the displacement of image structures between observations. **Brightness temperature (BT)** expresses measured radiance as an equivalent blackbody temperature in kelvin. Radiometric evolution describes changes in BT. The scientific question is how the training objective changes the predictive information in the learned representations. An observed residual from an advective forecast combines radiometric evolution, errors in estimated motion and observational variability.

**Persistence** repeats the final observed image; **Advection** extrapolates estimated image motion. Both are deterministic forecast algorithms. The historical grid shares 512 validation clips from 17 September 2020. The 131,072-clip series has 512 newly sampled validation clips from that date and an additional 10,240-clip validation. Distinct validation populations are identified in the learning-curve legends.

The [dataset study](study_seviri_ssl_dataset.ipynb) describes spatial origins, observation dates, sampling and normalization. The [cloud and convection probing study](study_seviri_cloud_mask_probe.ipynb) evaluates CMA, CT and RDT targets on held-out periods. The earlier forecasting and radiometric probes are retained in a [supplementary notebook](study_seviri_ssl_radiometric_probes.ipynb).

## Running and refreshing the study

Select **Python (deep_convection)** and execute the cells in order. The configuration captures the saved progress of every registered run. Comparisons use validation reports from explicit completed epochs. Re-executing the cells refreshes that snapshot.

The default presentation uses figures and explanatory text. **SHOW_TABLES = True** additionally displays the numerical tables; CSV exports retain their exact values in either mode. **USE_WIDGETS = True** enables metric and variability selectors for the channel figures. The notebook reads saved metrics and images. Observation times are UTC; the snapshot timestamp uses Europe/Paris.

In [ ]:
from pathlib import Path
import importlib, json, sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/emma_gpm").is_dir())
sys.path.insert(0, str(ROOT / "src"))
SERIES = ROOT / "results/ssl_scaling_20261025"
SERIES_PATHS = [SERIES, ROOT / "results/ssl_extension_20261026", ROOT / "results/ssl_base_32768_20261026"]
LEARNING_SERIES_PATHS = SERIES_PATHS + [ROOT / "results/ssl_factorial_20261028"]
OUTPUT = ROOT / "results/training_study_20261110"
OUTPUT.mkdir(exist_ok=True)
USE_WIDGETS = False
SHOW_TABLES = False
if "monitor" in globals():
    monitor.close()
from emma_gpm import ssl_study, ssl_dashboard
study = importlib.reload(ssl_study)
views = importlib.reload(ssl_dashboard)
loaded_studies = [study.load_study(path) for path in SERIES_PATHS if (path / "scaling_plan.json").exists()]
SNAPSHOT = study.merge_studies(*loaded_studies)
learning_studies = loaded_studies + [study.load_study(path) for path in LEARNING_SERIES_PATHS
                                   if path not in SERIES_PATHS and (path / "scaling_plan.json").exists()]
LEARNING_SNAPSHOT = study.merge_studies(*learning_studies)
GRID_COMPLETION = ROOT / "results/ssl_grid_completion_20261103"
if (GRID_COMPLETION / "scaling_plan.json").exists():
    LEARNING_SNAPSHOT = study.add_completed_experiments(LEARNING_SNAPSHOT, study.load_study(GRID_COMPLETION))
# The main notebook presents SSL training. Historical probes have a separate notebook.
LEARNING_SNAPSHOT = {**LEARNING_SNAPSHOT, "runs": {
    key: run for key, run in LEARNING_SNAPSHOT["runs"].items()
    if run["stage"]["kind"] == "pretraining"}}
SNAPSHOT = LEARNING_SNAPSHOT
GEOGRAPHIC_SERIES = next((ROOT / name for name in (
    "results/ssl_131072_verified_20261102", "results/ssl_131072_20261102")
    if (ROOT / name / "scaling_plan.json").exists()), ROOT / "results/ssl_131072_verified_20261102")
GEOGRAPHIC_SNAPSHOT = (study.load_study(GEOGRAPHIC_SERIES)
                       if (GEOGRAPHIC_SERIES / "scaling_plan.json").exists() else None)

LEONARDO_TRAINING = Path("/data1/deep_convection_results/leonardo_base_future_131072_20261108/training")
if GEOGRAPHIC_SNAPSHOT and (LEONARDO_TRAINING / "progress.json").exists():
    GEOGRAPHIC_SNAPSHOT = study.attach_distributed_training(GEOGRAPHIC_SNAPSHOT, LEONARDO_TRAINING)

VALIDATION_STUDIES = {"Historical validation · 512 clips": LEARNING_SNAPSHOT}
if GEOGRAPHIC_SNAPSHOT:
    VALIDATION_STUDIES["Resampled validation · 512 clips"] = GEOGRAPHIC_SNAPSHOT

def table(frame):
    if not SHOW_TABLES:
        return
    if frame.empty:
        print("This comparison has no saved results in the selected snapshot.")
        return
    formats = {column: "{:.0f}" if pd.api.types.is_integer_dtype(frame[column]) else "{:.2f}"
               for column in frame.select_dtypes(include="number").columns}
    for column in frame.columns:
        if any(term in column.lower() for term in ("epoch", "clips", "updates", "models", "budget")):
            if pd.api.types.is_numeric_dtype(frame[column]):
                formats[column] = "{:.0f}"
        if "(%)" in column or column in ("Mean BT (K)", "BT standard deviation (K)"):
            formats[column] = "{:.1f}"
    display(frame.style.format(formats, na_rep="—").hide(axis="index"))

def figure(fig, filename):
    if fig is not None:
        fig.savefig(OUTPUT / filename, dpi=120, bbox_inches="tight")
        display(fig)
        plt.close(fig)

display(Markdown(f"**Snapshot:** {LEARNING_SNAPSHOT['captured_at'][:19].replace('T', ' ')} Europe/Paris · "
                 f"**Learning-curve series:** {len(learning_studies)} · **Figures and CSV files:** `{OUTPUT.relative_to(ROOT)}`"))

## Available training results

The historical grid contains forty completed SSL experiments: four objectives, two encoder sizes and five dataset sizes. Small Future and Joint at 16,384 clips retain their original 75-epoch schedules; the other historical runs have 150 epochs. Each saved epoch has a committed checkpoint and validation report.

Leonardo completed Base Future on 131,072 clips for 150 epochs and 32,400 optimizer updates. Small MVM at that size remains suspended after three epochs. Other registered 131,072-clip runs have no saved trajectory. The inventory below counts SSL experiments and exports their individual status to `training_status.csv`.

In [ ]:
progress_frames = [study.progress_frame(LEARNING_SNAPSHOT).assign(Protocol="Historical spatial grid")]
progress_studies = [LEARNING_SNAPSHOT]
if GEOGRAPHIC_SNAPSHOT:
    progress_frames.append(study.progress_frame(GEOGRAPHIC_SNAPSHOT).assign(Protocol="Pixel-resolution geographic sampling"))
    progress_studies.append(GEOGRAPHIC_SNAPSHOT)
progress = pd.concat(progress_frames, ignore_index=True)
table(progress)
progress.to_csv(OUTPUT / "training_status.csv", index=False)
completed = sum(run["complete"] for snapshot in progress_studies for run in snapshot["runs"].values())
display(Markdown(f"**{completed} of {len(progress)} runs completed.** Saved epochs: {progress['Saved epochs'].sum():,} of {progress['Target epochs'].sum():,}."))

## Forecast accuracy across the available training sets

**RMSE** is the root mean squared difference between predicted and observed brightness temperature, in kelvin. **Global RMSE** pools squared errors over the 512 validation clips, all pixels, seven channels and both lead times before taking the square root. **MAE** averages absolute temperature errors over the same domain. Lower values indicate smaller errors.

The bars compare Future, Joint and Residual, wherever available, with Persistence and Advection. Each architecture and training size uses the latest completed epoch shared by its available forecasting objectives. The panel titles identify that epoch. Earlier 75-epoch experiments therefore retain their original comparison budget when paired with later 150-epoch runs. All panels share a vertical scale. **RMSE reduction vs Advection** expresses the reduction relative to the advective RMSE; positive values indicate improvement.

These panels summarize each completed training schedule. The later differential curves compare validation loss throughout the shared optimizer-update history. MVM's reconstruction task has a dedicated evaluation section. These forecast bars use the historical common validation; the 131,072-clip learning curves and its two validation populations appear below.

In [ ]:
forecast_selection, forecast_coverage = study.select_budget(SNAPSHOT, "pretraining")
forecast_scores = study.metrics_frame(SNAPSHOT, forecast_selection)
forecast_summary = study.forecast_summary(forecast_scores)
table(forecast_coverage)
table(forecast_summary[["Training clips", "Encoder", "Model", "Epoch", "RMSE (K)", "MAE (K)", "RMSE reduction vs Advection (%)"]])
figure(study.overview_figure(forecast_scores), "forecast_overview.png")
forecast_summary.to_csv(OUTPUT / "forecast_summary.csv", index=False)
forecast_scores.to_csv(OUTPUT / "forecast_channel_metrics.csv", index=False)
display(Markdown(study.interpretation(SNAPSHOT, forecast_summary)))

## Observations, sampling units and forecast times

A **clip** is a sampling unit identified by its first timestamp and the origin of a spatial crop. Each clip contains 16 observations separated by five minutes, seven thermal channels and a 224 × 224 pixel crop. Its array shape is `(time, channel, y, x) = (16, 7, 224, 224)`. The elapsed time between its first and last observation is 75 minutes. Every selected clip contains finite observations at all required timestamps, channels and pixels, within the accepted brightness-temperature interval of 100–400 K. The reader uses the coordinate axes to orient each crop with north at the top and west on the left.

The channel order is **WV_062, WV_073, IR_087, IR_097, IR_108, IR_120, IR_134**. These identifiers denote the water-vapour and infrared channels at nominal wavelengths 6.2, 7.3, 8.7, 9.7, 10.8, 12.0 and 13.4 μm. In this notebook, **IR108** is the shorter name for IR_108.

The first ten frames form the historical input, spanning 45 minutes. The timestamp of the tenth frame is the **forecast reference time**. Each forecasting model receives this history and predicts two images, at **+15 and +30 minutes** relative to the reference time. In the stored clip, the historical frames have indices 0–9 and the targets have indices 12 and 15. The intermediate future observations remain available in the archive.

Masked reconstruction uses the same ten historical frames. This common input interval makes the information available to the four encoders comparable. All 16 stored frames contribute to clip validation and to the training-set normalization statistics.

A single global mean and standard deviation, estimated from all channels and frames of the training clips, normalize both training and validation. Reported errors use Kelvin units. The [dataset study](study_seviri_ssl_dataset.ipynb) contains the sampling distributions, geographic checks and normalization statistics.

## Encoder architectures and four learning objectives

The grid includes **VideoMAE-Small**, with feature dimension 384 and 12 Transformer blocks. Its input is divided into **tubelets**, each containing two frames, a 16 × 16 pixel region and all seven channels. The ten-frame history produces five temporal positions on a 14 × 14 spatial grid, giving 980 tokens.

The forecasting head combines the five temporal feature vectors at each spatial position, applies spatial convolutions and reconstructs two 224 × 224 images with seven channels each. Within each architecture, the objectives start from identical randomly initialized encoder weights, and their forecasting heads share their initial weights. Forecast inputs are restricted to observations at or before the forecast reference time.

### MVM — masked video modeling

**MVM** learns to reconstruct hidden historical brightness temperatures. Tube masking hides 176 of the 196 spatial patch positions throughout the history, covering approximately 90% of the tokens. The encoder receives the visible tokens, and a four-block decoder with feature dimension 192 reconstructs hidden values.

The decoder evaluates 50% of the token positions, using a spatial pattern whose position changes along the temporal sequence. The reconstruction loss is the mean squared error on positions that are both hidden from the encoder and selected for decoding. Training masks vary between presentations of a clip; validation uses a fixed mask per clip.

### Future — causal future-frame prediction

**Future** receives the complete historical input and predicts both future frames directly. The encoder and forecasting head are optimized together using mean squared error across clips, both lead times, all seven channels and all pixels. The targets are the observed images at +15 and +30 minutes.

### Joint — masked reconstruction and future prediction

**Joint** shares one encoder between a masked reconstruction pass and a complete-history forecasting pass. Its objective adds the reconstruction and forecasting losses with equal weights. The reconstruction decoder and forecasting head have separate parameters. The learning curves show their weighted sum as one Joint objective. The saved components also support the historical-reconstruction and forecast evaluations.

### Residual — advection-residual prediction

**Residual** predicts a correction to a fixed advective forecast. The target correction is the observed future image minus that advective forecast. Adding the predicted correction to the advective image produces the final forecast, whose mean squared error drives training. The correction can account for radiometric evolution, spatially varying motion and errors in the estimated transport.
**VideoMAE-Base** increases the encoder feature dimension from 384 to 768 and uses 12 attention heads per block. It retains 12 Transformer blocks; its four-block reconstruction decoder has feature dimension 384 instead of 192. The forecasting head retains width 192. Small and Base have now both completed all four objectives at the five historical dataset sizes, allowing architecture comparisons on identical training and validation manifests. Base Future and Small MVM also have saved trajectories at 131,072 clips.

## Forecast references: Persistence and Advection

**Persistence** copies the final observed image to each future lead time. Its forecast accuracy measures the value of retaining the latest observation.

**Advection** estimates a single horizontal displacement from the last two historical IR108 images using phase correlation, an image-alignment procedure in the Fourier domain. It assigns the same motion vector to every pixel in the crop. Each displacement component is bounded to ±12 pixels per five-minute interval. The algorithm then transports the final image at constant velocity: three displacement steps for +15 minutes and six for +30 minutes. All seven thermal channels use that translation. The algorithm estimates motion independently for each input clip and uses a fixed extrapolation rule.

The **advection support** contains pixels whose translated source positions lie inside the crop. At the other positions, the forecast retains the last observed temperature at the corresponding pixel. This boundary rule supplies a complete forecast and a support mask for evaluation.

**Residual** is the trained experiment associated with this reference: its neural network predicts a temperature correction, which is added to the advected image. The deterministic Advection forecast and the trained Residual forecast appear as separate legend entries because they produce two different predictions. Their difference measures the contribution of the learned correction, including compensation for radiometric evolution, cloud deformation and errors in the uniform-motion approximation.

## Learning trajectories by optimizer updates

Each figure follows **one learning objective** across architectures and dataset sizes. **MVM** measures squared error on decoder-selected hidden historical pixels. **Future** measures squared error on the two forecast images. **Residual** measures squared error on its final forecast, assembled from advection and the learned correction. **Joint** adds historical reconstruction and forecast error with weights equal to one. Its curves represent that complete sum. Every curve uses the run's original normalization to express objective loss in **K²**.

**Colour identifies training-set size**, from 4,096 to 131,072 clips, with the same colours across objectives. **Solid lines identify Small; dashed lines identify Base.** The upper panels show training and validation with logarithmic axes and independent vertical ranges. The wide lower panel shows the same validation observations on a linear vertical axis, enlarging the range around the lower losses. Its upper limit is based on the 85th percentile of losses after the first quarter of each recorded trajectory; all recorded peaks remain visible in the upper panel. Optimizer updates use a logarithmic horizontal axis throughout. Each star marks the lowest saved validation loss of that run, at the first update where it was attained. A hollow circle marks the latest saved epoch of an incomplete run, including the suspended three-epoch Small MVM run.

**Thicker violet curves identify the 131,072-clip series. Diamonds mark its 512-clip resampled validation; squares mark Base Future evaluated on the 10,240-clip extended validation. Both evaluations appear on the same validation axes, with their sample counts in the legend.** The ordinary lines use the historical validation. Both 512-clip populations come from 17 September 2020; the new population uses randomly positioned crops whose entire footprint lies inside 20°W–40°E, 30–70°N. The training origins were also resampled. The differential plots use the shared historical validation. The [dataset notebook](study_seviri_ssl_dataset.ipynb) explains the sampling changes.

All historical experiments use **32 clips per optimizer update**. Their five sizes give 128, 256, 512, 1,024 and 2,048 updates per epoch. The local 131,072-clip Small MVM run also uses batch 32 and has 4,096 updates per epoch. **Leonardo Base Future uses a global batch of 608 clips**, distributed as 152 on each of four GPUs, and records 216 updates per epoch. Its 150 epochs end at 32,400 updates. Most updates therefore average 19 times as many clip presentations as a batch-32 update; the final batch of an epoch can be smaller. Equal update counts describe equal numbers of parameter changes, while batch size determines the observations contributing to each change. Architecture, objective, hardware and input throughput determine elapsed time per update.

AdamW uses a constant learning rate of 0.00015 and weight decay 0.05. The original local execution accumulated two batches of 16; subsequent local execution uses batches of 32. Execution settings and checkpoint lineage are recorded for each run.

Training loss averages batches while parameters evolve during the epoch. Validation evaluates the saved end-of-epoch weights, with fixed reconstruction masks where applicable. Every committed epoch is shown, including temporary error peaks. `learning_trajectories.csv` and `objective_validation_minima.csv` include the validation-population label and manifest identifier, so every exported value retains its evaluation context.

Base Future uses the same saved weights for its two validation curves. The 512-clip population is evaluated every epoch and selects the checkpoint; the extended population is evaluated every ten epochs and contains 512 clips from each of twenty dates between January and June 2022. Both curves show mean squared forecast error in K². Their separate populations are identified in the legend.

In [ ]:
learning_histories = []
minima_frames = []
for population, snapshot in VALIDATION_STUDIES.items():
    minima_frames.append(study.objective_minima(snapshot).assign(
        validation_population=population, validation_sha256=snapshot["validation_sha256"]))
    for run in snapshot["runs"].values():
        if run["stage"]["kind"] != "pretraining" or not run["epochs"]:
            continue
        data = snapshot["cases"][run["case"]]
        history = study.native_loss_history(snapshot, run)
        history = history[["epoch", "optimizer_steps", "train_objective_k2", "validation_objective_k2"]].copy()
        history = history.assign(objective=run["name"], encoder=data["encoder"], training_clips=data["train_samples"],
            validation_population=population, validation_sha256=snapshot["validation_sha256"])
        learning_histories.append(history)
pd.concat(learning_histories, ignore_index=True).to_csv(OUTPUT / "learning_trajectories.csv", index=False)
additional = dict(list(VALIDATION_STUDIES.items())[1:])
extra_validation = []
if GEOGRAPHIC_SNAPSHOT and (LEONARDO_TRAINING / "validation_extended").exists():
    extended_history = study.distributed_validation_history(LEONARDO_TRAINING)
    extended_history.to_csv(OUTPUT / "base_future_131072_extended_validation.csv", index=False)
    extra_validation.append(dict(objective="future", encoder="Base", training_clips=131072,
        label="Base · 131,072 · 10,240 validation clips", history=extended_history))
for objective, fig in study.objective_learning_figures(LEARNING_SNAPSHOT, additional_studies=additional,
        extra_validation=extra_validation, validation_zoom=True).items():
    figure(fig, f"learning_{objective}.png")
objective_minima = pd.concat(minima_frames, ignore_index=True)
objective_minima.to_csv(OUTPUT / "objective_validation_minima.csv", index=False)
table(objective_minima)


### Latest error and optimization history

The learning curves retain the full optimization history. `optimization_history.csv` records each run's latest validation RMSE, minimum recorded value and epoch of that minimum. This RMSE summary complements the native-objective minima marked in the learning curves; for Joint, the star selects the minimum of its summed reconstruction and forecast loss. For MVM the quantity is historical-reconstruction error; for Future, Joint and Residual it is forecast error. The history summary includes the same full set of runs as the learning curves. The forecast overview uses the latest common epoch of the available objectives within each architecture and dataset size, as identified by its panel titles. Minima along a trajectory describe repeated evaluation on the same validation sample.

In [ ]:
trajectories = pd.concat([study.trajectory_summary(snapshot).assign(validation_population=population)
                          for population, snapshot in VALIDATION_STUDIES.items()], ignore_index=True)
table(trajectories)
trajectories.to_csv(OUTPUT / "optimization_history.csv", index=False)
late_windows = []
for (case, name), run in SNAPSHOT["runs"].items():
    if case != "n32768" or name not in ("future", "joint") or len(run["epochs"]) < 60:
        continue
    history = pd.DataFrame(run["epochs"])
    std = SNAPSHOT["cases"][case]["stats"]["global_std_k"]
    for label, window in (("Previous 30 epochs", history.iloc[-60:-30]), ("Last 30 epochs", history.iloc[-30:])):
        late_windows.append({"Training clips": 32768, "Encoder": "Base", "Model": views.LABELS[name],
            "Window": label, "First epoch": int(window.epoch.iloc[0]), "Last epoch": int(window.epoch.iloc[-1]),
            "Mean validation RMSE (K)": window.validation_rmse_k.mean(),
            "Median validation RMSE (K)": window.validation_rmse_k.median(),
            "Training forecast RMSE (K)": window.train_future.mean()**.5 * std})
    a, b = late_windows[-2:]
    display(Markdown(f"**Base / 32,768 · {views.LABELS[name]}:** between epochs {a['First epoch']}–{a['Last epoch']} "
        f"and {b['First epoch']}–{b['Last epoch']}, median validation forecast RMSE changes from "
        f"**{a['Median validation RMSE (K)']:.2f} to {b['Median validation RMSE (K)']:.2f} K**, "
        f"while training forecast RMSE changes from {a['Training forecast RMSE (K)']:.2f} to "
        f"{b['Training forecast RMSE (K)']:.2f} K. These summaries describe late optimization on the same validation day."))
pd.DataFrame(late_windows).to_csv(OUTPUT / "late_training_windows.csv", index=False)


## Separating model capacity and training-set size throughout optimization

The following figures compare **validation loss trajectories within the same training objective**, using the original global temperature scales to express every value in K². Each figure has two views. **Current validation** subtracts the reference run's validation loss from the candidate run's loss at the same saved optimizer update. **Best validation reached so far** subtracts their respective lowest validation losses observed up to that update. Each running minimum uses every earlier saved evaluation of its own run, including evaluations between two points shared by the pair. It can change only when that run reaches a new minimum. This second view follows how the best available result develops throughout optimization.

### Model capacity at a fixed dataset

For **Base − Small**, both encoders use the same training manifest and objective. Each coloured curve identifies a dataset size: 4,096, 8,192, 16,384, 32,768 or 65,536 clips. A **negative difference favours Base**; a positive difference favours Small. Zero means equal validation loss under the displayed criterion. These comparisons isolate the architecture choice under the recorded common data and optimization settings. Architecture also changes GPU work per update.

### Dataset size at a fixed architecture

For both **Small** and **Base**, each curve subtracts the 4,096-clip run from a larger dataset run: 8,192 − 4,096, 16,384 − 4,096, 32,768 − 4,096 and 65,536 − 4,096. The smallest available dataset for each architecture supplies its reference; the completed grid now includes 4,096 clips for both. Every pair retains the same architecture and objective. A **negative difference favours the larger dataset**. Colours identify the larger dataset, consistently with the absolute loss plots; legend labels state both dataset counts. Available objectives determine which pairs appear.

### Shared history and interpretation of the minima

Each point uses an **exact update count saved by both members of that pair**. Connecting segments guide the eye between these observations. Each curve spans its own shared history and ends at its last shared observation. The 4,096-clip runs end at 19,200 updates, so dataset contrasts for either architecture extend to at most that count; the 65,536 − 4,096 contrast ends at 18,432, the final shared saved update. The absolute loss curves above retain the subsequent training history of the larger datasets. The Base − Small curves extend farther where both runs have longer histories.

The horizontal axis is logarithmic. The difference axis is linear and includes zero, with each panel's range fitted to its displayed differences. Temporary validation peaks remain visible in the current-loss view. The best-so-far view retains the best score attained through those oscillations. The stars in the absolute curves summarize each run's complete available history; a difference between such final minima also includes the different maximum training budgets. Longer runs provide more updates and more validation evaluations from which to select a minimum. Validation is saved once per epoch, so smaller datasets also provide more validation evaluations at a fixed update count. The best-so-far curves inherit this evaluation cadence. Every running-minimum comparison uses information available up to its plotted update.

The CSV exports `architecture_loss_differences.csv` and `dataset_loss_differences.csv` retain both runs' losses, cumulative minima, epoch numbers and update counts, together with the signed differences. Their direction is always candidate minus reference. The earlier forecast overview and channel diagnostics retain their stated reference configurations.

These differences use the shared historical validation. The absolute plots include the resampled 131,072-clip population with its own markers. A paired difference involving that population will require saved evaluations of both models on a common validation manifest.

In [ ]:
for comparison in ("architecture", "dataset"):
    differences = study.learning_differences(LEARNING_SNAPSHOT, comparison)
    differences.to_csv(OUTPUT / f"{comparison}_loss_differences.csv", index=False)
    heading = "Model capacity: Base − Small" if comparison == "architecture" else "Dataset size: fixed encoder architecture"
    display(Markdown(f"### {heading}"))
    for objective, fig in study.learning_difference_figures(differences).items():
        figure(fig, f"{comparison}_difference_{objective}.png")

## Errors by channel: all training configurations together

Each row compares the available models and dataset sizes for **one spectral channel**, with +15 minutes on the left and +30 minutes on the right. Bar groups identify architecture and number of training clips; colours identify Future, Joint and Residual, or the frozen encoders when selecting the probe family. The two lead times share their vertical limits within each row. **Persistence** and **Advection** appear as dashed reference lines because they use the same validation observations for every training configuration.

The first row, **All channels**, pools squared errors across all seven channels before taking the square root for RMSE. The remaining rows isolate the individual channels. This makes it possible to compare Small and Base directly within a channel and then assess whether that difference persists in the pooled result. The selected final epochs follow the overview comparison; the preceding differential curves compare optimization histories at shared update counts.

The default metric is RMSE in kelvin. **Supported RMSE** uses pixels whose advected source positions remain inside the crop. **Gradient MAE** measures error in adjacent-pixel temperature differences, in K/pixel. **Local-minimum MAE** evaluates temperature errors at observed minima in 3 × 3 neighbourhoods, including ties. **Minimum-temperature MAE** compares the coldest predicted and observed temperature in each crop. **Contrast MAE** compares their 95th-to-5th percentile temperature ranges.

**Reference-time tendency MAE** expresses the forecast temperature error per minute of lead time. **Interval tendency MAE** evaluates the first and second consecutive 15-minute changes separately. For these MAE metrics, All channels is the equal-weight mean of the seven channel values; local-minimum counts can differ between channels. RMSE always uses pooled squared errors.

`FAMILY`, `METRIC` and `VARIABILITY` select which saved results are shown. Numerical values and comparison coverage are exported with the figures.

In [ ]:
FAMILY = "pretraining"
METRIC = "rmse_k"
VARIABILITY = "all"  # all, low, medium, high

def show_channels(family, metric, variability):
    selected, coverage = study.select_budget(SNAPSHOT, family)
    scores = study.metrics_frame(SNAPSHOT, selected, variability)
    table(coverage)
    coverage.to_csv(OUTPUT / f"channel_coverage_{family}.csv", index=False)
    if scores.empty:
        print("Channel comparisons are awaiting completed validation results.")
    else:
        figure(study.channel_training_figure(scores, metric), f"channels_{family}_{metric}_{variability}.png")
        study.aggregate_channels(scores).to_csv(OUTPUT / f"channel_comparison_{family}_{variability}.csv", index=False)
        table(study.forecast_summary(scores)[["Training clips", "Model", "Epoch", "RMSE +15 min (K)", "RMSE +30 min (K)"]])

if USE_WIDGETS:
    import ipywidgets as widgets
    metric = widgets.Dropdown(options=[(label, key) for key, label in views.METRICS.items()], value=METRIC, description="Metric:")
    variability = widgets.Dropdown(options=["all", "low", "medium", "high"], value=VARIABILITY, description="Variability:")
    channel_output = widgets.Output()
    def refresh_channels(*_):
        with views.widget_output(channel_output):
            show_channels(FAMILY, metric.value, variability.value)
    for control in (metric, variability):
        control.observe(refresh_channels, names="value")
    refresh_channels()
    display(widgets.VBox([widgets.HBox([metric, variability]), channel_output]))
else:
    show_channels(FAMILY, METRIC, VARIABILITY)

## Does the forecast improvement also cover rapidly changing scenes?

A single global RMSE mixes relatively stable cloud scenes with scenes that change strongly during the input history. The purpose of this diagnostic is to ask **whether a learned model improves on simple transport in both situations**, especially in the rapidly changing scenes relevant to the dynamics question in the project. A model could obtain a useful average score mainly by predicting the stable cases well. Comparing the groups reveals where its improvement occurs and which observations deserve closer inspection or broader sampling.

Each clip's **historical variability score** is the mean absolute IR108 temperature change between successive five-minute observations, averaged over all crop pixels and the nine historical frame pairs. Fixed thresholds of approximately **0.75 and 1.44 K**, estimated from the 4,096-clip training set, divide the validation clips into Low, Medium and High. The same groups are used for every model and training size. The score is calculated from the ten input frames available before forecasting.

The bars now show **percentage reduction in RMSE relative to Advection within each group**. Positive values show added forecast accuracy beyond the estimated uniform transport; zero gives the same RMSE as Advection. A larger positive value in High means that the learned forecast contributes more relative improvement for the rapidly changing input sequences. Absolute errors are retained in the CSV export and in the examples described below. The comparison uses the common final checkpoints from the overview.

For example, if two methods have similar global RMSE but one improves more in High, that method warrants closer examination for rapidly evolving structures. If improvement is concentrated in Low, the aggregate score mainly supports forecasting of stable scenes. Historical changes combine displacement of cloud edges and changes in temperature and shape. The present grouping selects observational variability; identifying convective growth specifically requires a target describing that physical process.

In [ ]:
conditional = study.variability_summary(SNAPSHOT, forecast_selection)
table(conditional[["Variability", "Validation clips"]].drop_duplicates())
figure(study.variability_figure(conditional, "RMSE reduction vs Advection (%)"), "historical_variability.png")
table(conditional[["Training clips", "Variability", "Model", "RMSE (K)", "RMSE reduction vs Advection (%)"]])
conditional.to_csv(OUTPUT / "forecast_by_variability.csv", index=False)
group_sizes = conditional[["Variability", "Validation clips"]].drop_duplicates()
display(Markdown("**Validation groups:** " + "; ".join(f"{row.Variability}: {row['Validation clips']:,} clips" for _, row in group_sizes.iterrows()) + "."))
example = conditional[(conditional["Training clips"] == 32768) & (conditional.Variability == "High") & (conditional.Encoder == "Base")]
if set(["Advection", "Joint"]).issubset(set(example.Model)):
    values = example.set_index("Model")
    display(Markdown(f"For **Joint Base / 32,768**, the High group's RMSE is **{values.loc['Joint', 'RMSE (K)']:.2f} K**, "
        f"compared with **{values.loc['Advection', 'RMSE (K)']:.2f} K** for Advection: "
        f"a **{values.loc['Joint', 'RMSE reduction vs Advection (%)']:.1f}%** reduction on those same validation clips."))

## Temperature errors and errors in cloud structure

Each point combines **two validation errors for one forecast method, one training size, one channel and one lead time**. The horizontal coordinate is image RMSE: the overall mismatch between predicted and observed temperatures, in kelvin. The vertical coordinate is **Gradient MAE**: the mean absolute error in the temperature difference between adjacent pixels, calculated horizontally and vertically, in K/pixel. Both coordinates pool all 512 validation clips. Colours identify methods; circles denote +15 minutes and squares +30 minutes. All panels share their horizontal and vertical limits.

The gradient error describes how well the prediction reproduces local temperature changes across cloud boundaries and internal structures. For example, a forecast that adds the same 2 K offset to every pixel has a temperature error while preserving all neighbouring-pixel differences. A spatially smoothed forecast can reduce some temperature errors while weakening cloud-edge gradients. Errors in the location of a sharp boundary can also increase gradient error. These examples explain why the two coordinates supply complementary information.

A point farther left has lower image RMSE; a point lower down has lower gradient error. Moving toward the lower-left improves both quantities. If one method lies left of another but higher, their ordering depends on the aspect of forecast quality being assessed. The graph places two diagnostics side by side for each prediction; it represents a comparison across methods and lead times.

The exported structural metrics add errors at observed local minima, the coldest temperature in each crop, thermal contrast and the change between consecutive forecast times. These quantities help distinguish temperature accuracy, boundary preservation and cold-extreme preservation. `STRUCTURE_CHANNEL` selects the thermal channel.

In [ ]:
STRUCTURE_CHANNEL = "IR_108"
figure(study.structure_figure(forecast_scores, STRUCTURE_CHANNEL), "radiometry_and_gradients.png")
structural = forecast_scores[forecast_scores.channel == STRUCTURE_CHANNEL].copy()
structural["model"] = structural.model.map(views.LABELS)
columns = ["training_clips", "model", "horizon_minutes", "rmse_k", "gradient_mae_k_per_pixel", "local_minima_mae_k", "minimum_temperature_mae_k", "contrast_p95_p05_mae_k", "interval_tendency_mae_k_per_min"]
structural = structural[columns].rename(columns={"training_clips": "Training clips", "model": "Model", "horizon_minutes": "Lead (min)", **views.METRICS})
table(structural)
structural.to_csv(OUTPUT / "ir108_structural_metrics.csv", index=False)

for size, rows in forecast_scores[forecast_scores.channel == STRUCTURE_CHANNEL].groupby("training_clips"):
    future = rows[(rows.model == "future") & (rows.horizon_minutes == 30)]
    advection = rows[(rows.model == "advection") & (rows.horizon_minutes == 30)]
    if not future.empty and not advection.empty:
        a, b = future.iloc[0], advection.iloc[0]
        display(Markdown(f"At **{size:,} clips** and +30 minutes, Future gives an {STRUCTURE_CHANNEL} image RMSE "
            f"of {a.rmse_k:.2f} K, compared with {b.rmse_k:.2f} K for Advection. "
            f"The error in the coldest temperature within each crop is {a.minimum_temperature_mae_k:.2f} K "
            f"for Future and {b.minimum_temperature_mae_k:.2f} K for Advection. "
            "These two metrics assess average radiometric accuracy and preservation of cold extremes, respectively."))

## Historical reconstruction

This section evaluates the native masked-reconstruction task for **MVM and Joint**. The targets are observed temperatures in the ten historical input frames. Tube masking hides about 90% of spatial patch positions throughout that history. During validation, the decoder evaluates its selected half of the token positions, and the loss uses positions that are both hidden and selected. Masks are fixed per validation clip and shared by the compared models. The bar plot converts this error to **reconstruction RMSE in kelvin** with each dataset's global temperature scale. The x-axis identifies the architecture, training-set size and selected epoch. Bars appear for the trained reconstruction objectives at each size.

The spatial example shows a historical frame from a saved validation clip. **Observation** is the complete measured image. **Masked input** shows the visible observations and marks hidden regions in grey. **Reconstruction** fills those hidden regions with the decoder's predictions while retaining the original observations in visible regions. This combination of predicted and observed pixels is the reason the earlier figure used the term *composite reconstruction*. **Masked-pixel error** shows predicted minus observed temperature only at hidden pixels; positive values indicate a warmer reconstruction and negative values a colder one.

The saved spatial example reconstructs all tokens hidden from the encoder. The quantitative bar plot uses the decoder-selected hidden targets from the validation objective. The figure therefore displays reconstruction across the full mask, while the bars follow the objective recorded during training. All displayed frames precede the forecast reference time. The example selector chooses dataset, channel and historical frame.

In [ ]:
reconstruction, reconstruction_coverage = study.reconstruction_summary(SNAPSHOT)
figure(study.reconstruction_comparison_figure(SNAPSHOT), "historical_reconstruction.png")
reconstruction.to_csv(OUTPUT / "historical_reconstruction.csv", index=False)
study.size_comparison(SNAPSHOT, "reconstruction").to_csv(OUTPUT / "reconstruction_matched_budgets.csv", index=False)

RECONSTRUCTION_DATASET = "n8192"
queue = SNAPSHOT["cases"][RECONSTRUCTION_DATASET]["queue"]
catalog = study.read_json(queue / "analysis/examples/catalog.json")
if catalog:
    MVM_CLIP_INDEX = catalog["indices"][0]
    MVM_CHANNEL = "IR_108"
    HISTORY_FRAME = 0
    if USE_WIDGETS:
        display(views.reconstruction_dashboard(queue))
    else:
        figure(views.reconstruction_figure(queue, MVM_CLIP_INDEX, MVM_CHANNEL, HISTORY_FRAME), "mvm_reconstruction.png")
else:
    print("Spatial reconstruction examples are scheduled with the final analysis after the training series.")

## Saved forecast examples

Every completed training run has a saved validation figure for the first clip in the shared validation manifest. The figures below compare that same observation for Future, Joint and Residual at the selected training size. The model is named immediately above its figure; **neural** inside the saved figure denotes that model's forecast. Each figure includes the last input, the future observation, Persistence and Advection at both lead times. Brightness temperatures share the 180–330 K colour scale, with north at the top and west on the left.

The final analysis also saves a four-clip example catalog spanning the validation variability range, with reusable prediction arrays and error maps. The catalog is used by the interactive forecast and reconstruction views once it is available.

In [ ]:
EXAMPLE_DATASET = "n8192"
for model in ("future", "joint", "residual"):
    run = SNAPSHOT["runs"].get((EXAMPLE_DATASET, model))
    if run is None:
        continue
    path = run["directory"] / "forecast_latest.png"
    if run["complete"] and path.exists():
        display(Markdown(f"**{views.LABELS[model]} · {SNAPSHOT['cases'][EXAMPLE_DATASET]['train_samples']:,} training clips · epoch {run['completed_epochs']}**"))
        display(Image(filename=str(path), width=1100))
    else:
        print(f"{views.LABELS[model]}: the final-epoch forecast image is awaiting completion.")

## Computational measurements

The separate [performance notebook](profile_seviri_ssl_performance.ipynb) records data preparation, GPU execution, validation, checkpoint timing, memory and CUDA activity. Its complete-epoch measurements compare the original and revised input pipelines. The exported performance comparison records those measurements, whose timings exclude initialization and profiler-recorded epochs. The current complete-epoch duration includes checkpoint persistence; the previous column sums the two components recorded in the historical logs.

The scientific comparison above retains each run's registered learning rate, effective batch and training budget. Execution revisions and their checkpoint lineage are recorded with the series. Joint's extra reconstruction pass contributes additional computational work per optimizer update.

In [ ]:
performance_path = SERIES / "execution_revisions/clip_files_v1/comparison.json"
if performance_path.exists():
    performance = study.read_json(performance_path)
    measured = pd.DataFrame(performance["timing_rows"])
    formats = {column: "{:.0f}" for column in measured.columns if column.endswith("(s)")}
    measured.to_csv(OUTPUT / "performance_comparison.csv", index=False)
    if SHOW_TABLES:
        display(measured.style.format(formats).hide(axis="index"))

## Reproducibility and saved products

The historical snapshot reads `ssl_scaling_20261025`, `ssl_extension_20261026`, `ssl_base_32768_20261026`, `ssl_factorial_20261028` and `ssl_grid_completion_20261103` under `results/`. The separate geographic source reads the prepared 131,072-clip plan and the downloaded Leonardo reports. Each source retains its own validation manifest and original global normalization. The loader checks registered manifest identity and the nested historical sampling before reading saved epoch metrics. The plotting overlay keeps these sources distinct; paired comparisons require a common validation manifest.

Current figures and CSV exports are written to `results/training_study_20261110/`. Every loss trajectory uses committed epochs. Native objectives are converted to K²; forecast RMSE pools squared errors across channels and lead times before taking the square root. No model inference is required to refresh these saved-result plots. Re-executing the notebook updates its snapshot.

The [dataset study](study_seviri_ssl_dataset.ipynb) documents the observation periods, crop origins and normalization. The [CMA/CT/RDT probing study](study_seviri_cloud_mask_probe.ipynb) presents the newer semantic and temporal targets on independent probe periods. Earlier radiometric probes and temporal interventions remain reproducible in the [supplementary study](study_seviri_ssl_radiometric_probes.ipynb), with their original saved products.

The [project specification](../handoff_codex_ssl_seviri.md) defines the scientific questions, and [SSL_SEVIRI.md](../SSL_SEVIRI.md) records preparation, training and monitoring commands.